In [ ]:
import torch, random, io, sys, warnings
import os, numpy as np, pandas as pd, pyreadr
from tqdm import tqdm
import sys, os
sys.path.append(os.path.abspath(".."))

from cpd_model import parse_args, learn_one_seq_penalty

warnings.filterwarnings("ignore")

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# ======================================================
# CONFIG
# ======================================================
REPS = 10
TRUE_CP = [100, 200]
Z_LIST = [2, 5, 10]
TOL = 10

# ======================================================
# PREPARE BASE ARGS (template)
# ======================================================
base_args = parse_args()
base_args.epoch = 150
base_args.K_dim = 2
base_args.decoder_lr = 0.01
base_args.decoder_iteration = 20
base_args.langevin_s = 0.2
base_args.langevin_K = 100
base_args.kappa = 0.8
base_args.penalties = [0.01, 0.05, 0.1, 1]
base_args.nu_iteration = 20
base_args.output_layer = [50, 50]
base_args.scale_delta = False
base_args.signif_level = 0.99
base_args.true_CP_full = TRUE_CP

# ======================================================
# MAIN LOOP
# ======================================================
records = []

GLOBAL_SEED = 1

for z_dim in Z_LIST:
    print(f"\n==============================")
    print(f" Running z_dim = {z_dim}")
    print(f"==============================")

    for rep in range(1, REPS + 1):

        print(f"\n--- Replicate {rep}/{REPS} ---")

        # ---------- seed ----------
        SEED = GLOBAL_SEED + rep
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

        # ---------- load data ----------
        Y = pyreadr.read_r(f"../real_data_sim/sim_dat_ult_5_{rep}.RDS")
        X = pyreadr.read_r(f"../real_data_sim/sim_x_ult_5_{rep}.RDS")

        Y_df = np.array(list(Y.values())[0])
        X_df = np.array(list(X.values())[0])

        # expand X
        X_rep = np.repeat(X_df[:, np.newaxis, :], 100, axis=1)
        Y = Y_df[:, :, 0:3]
        X = X_rep

        # ---------- build args ----------
        args = parse_args()
        args.__dict__.update(base_args.__dict__)

        args.z_dim = z_dim
        args.x_dim = X.shape[2]
        args.y_dim = Y.shape[2]
        args.num_time = X.shape[0]
        args.num_samples = X.shape[1]

        # ---------- tensors ----------
        x_input = torch.tensor(X, dtype=torch.float32).to(device)
        y_input = torch.tensor(Y, dtype=torch.float32).to(device)

        # ---------- split ----------
        odd_idx = range(1, args.num_time, 2)
        even_idx = range(0, args.num_time, 2)

        x_train = x_input[odd_idx].reshape(-1, args.x_dim)
        x_test  = x_input[even_idx].reshape(-1, args.x_dim)
        y_train = y_input[odd_idx].reshape(-1, args.y_dim)
        y_test  = y_input[even_idx].reshape(-1, args.y_dim)

        # ======================================================
        # penalty selection
        # ======================================================
        results_half = []

        for penalty in args.penalties:
            _stdout = sys.stdout
            # sys.stdout = io.StringIO()
            try:
                loss, pen = learn_one_seq_penalty(
                    args, x_train, y_train, x_test, y_test,
                    penalty=penalty, half=True
                )
            finally:
                sys.stdout = _stdout

            results_half.append([loss, pen])

        results_half = np.array(results_half)
        best_idx = np.argmin(results_half[:, 0])
        best_penalty = args.penalties[best_idx]

        print(f"[INFO] Best penalty = {best_penalty}")

        # ======================================================
        # full training
        # ======================================================
        _stdout = sys.stdout
        sys.stdout = io.StringIO()
        try:
            out = learn_one_seq_penalty(
                args,
                x_input.reshape(-1, args.x_dim),
                y_input.reshape(-1, args.y_dim),
                x_input.reshape(-1, args.x_dim),
                y_input.reshape(-1, args.y_dim),
                penalty=best_penalty,
                half=False
            )
            result = out[0]
        finally:
            sys.stdout = _stdout

        torch.cuda.empty_cache()

        # ======================================================
        # evaluation
        # ======================================================
        est_cp = np.array(result[5], dtype=int) if len(result[5]) > 0 else np.array([])
        true_cp = np.array(TRUE_CP)

        if len(est_cp) == 0:
            cover_rate = 0
            avg_dist = np.nan
            FP = 0
            FN = len(true_cp)
        else:
            dist_mat = np.abs(est_cp[:, None] - true_cp[None, :])
            min_dist_true = dist_mat.min(axis=0)
            min_dist_est  = dist_mat.min(axis=1)

            cover_rate = np.mean(min_dist_true <= TOL)
            avg_dist   = np.mean(min_dist_true)
            FP = np.sum(min_dist_est > TOL)
            FN = np.sum(min_dist_true > TOL)

        # ======================================================
        # record
        # ======================================================
        records.append({
            "z_dim": z_dim,
            "rep": rep,
            "best_penalty": best_penalty,
            "num_detected": len(est_cp),

            # core output
            "est_CP": str(list(est_cp)),
            "true_CP": str(TRUE_CP),

            # optional (keep if you want)
            "CE": result[0],   # count error
        })


# ======================================================
# SAVE RESULTS
# ======================================================
df = pd.DataFrame(records)
df.to_csv("cpd_zdim_experiment.csv", index=False)


[INFO] Using device: cuda

 Running z_dim = 2

--- Replicate 1/10 ---
Epoch   5 | Loss=609.502014 | Kurtosis=6.008619
Epoch  10 | Loss=582.325989 | Kurtosis=3.715803
Epoch  15 | Loss=563.663086 | Kurtosis=10.528452
Epoch  20 | Loss=553.645325 | Kurtosis=17.069744
Epoch  25 | Loss=547.518860 | Kurtosis=16.024055
Epoch  30 | Loss=544.611145 | Kurtosis=24.022284
Epoch  35 | Loss=544.368958 | Kurtosis=26.856068
Epoch  40 | Loss=545.350342 | Kurtosis=24.458593
Epoch  45 | Loss=544.685730 | Kurtosis=25.249449
Epoch  50 | Loss=544.124817 | Kurtosis=21.768063
Epoch  55 | Loss=543.755798 | Kurtosis=24.624069
Epoch  60 | Loss=543.209351 | Kurtosis=26.966942
Epoch  65 | Loss=544.505920 | Kurtosis=23.395683
Epoch  70 | Loss=543.420837 | Kurtosis=24.798763
Epoch  75 | Loss=544.037170 | Kurtosis=20.048094
Epoch  80 | Loss=544.144287 | Kurtosis=21.193283
Epoch  85 | Loss=543.850586 | Kurtosis=19.312199
Epoch  90 | Loss=543.827393 | Kurtosis=21.579588
Epoch  95 | Loss=543.780762 | Kurtosis=19.290735
E